# Banking AI Lab — exploratory transaction analysis

## Learning goals

Load the checked-in synthetic transaction dataset with the Stage 3 DataFrame types, inspect its stable schema without exposing row-level records, and practise six focused analyses covering data quality, distributions, grouped patterns, time, and rare-class metrics. Decimal text deliberately crosses into `float64` at this EDA boundary; domain and persistence code continue to use exact `Decimal`.

## Local and Colab data setup

Run locally from the repository root. In Colab, upload only the checked-in synthetic `transactions.csv` when the setup cell prompts you; the fallback copies that upload to the same relative `DATA_PATH`. This is a manual browser upload, so no network download is required.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 20260919
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DATA_PATH = Path("datasets/transactions.csv")

if not DATA_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Run from the repository root or use the Colab upload fallback."
        ) from exc
    uploaded = files.upload()
    if DATA_PATH.name not in uploaded:
        raise FileNotFoundError("Upload the checked-in synthetic transactions.csv file.")
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    DATA_PATH.write_bytes(uploaded[DATA_PATH.name])

In [ ]:
EXPECTED_COLUMNS = (
    "id",
    "user_id",
    "amount",
    "currency",
    "merchant",
    "category",
    "timestamp",
    "status",
    "is_fraud",
)
transactions = pd.read_csv(DATA_PATH, dtype="string", keep_default_na=False)
if tuple(transactions.columns) != EXPECTED_COLUMNS:
    raise ValueError("Unexpected transaction columns")
transactions["amount"] = pd.to_numeric(transactions["amount"], errors="raise").astype("float64")
currency_order = ("EUR", "GBP", "MDL", "USD")
if not set(transactions["currency"]) <= set(currency_order):
    raise ValueError("Unexpected currency value")
transactions["currency"] = transactions["currency"].astype(
    pd.CategoricalDtype(categories=currency_order, ordered=True)
)
category_order = sorted({str(value) for value in transactions["category"] if value})
transactions["category"] = (
    transactions["category"]
    .replace("", pd.NA)
    .astype(pd.CategoricalDtype(categories=category_order, ordered=True))
)
transactions["timestamp"] = pd.to_datetime(
    transactions["timestamp"], errors="raise", utc=True
).astype("datetime64[ns, UTC]")
fraud_labels = {"true": True, "false": False}
raw_fraud = transactions["is_fraud"].str.lower()
if not raw_fraud.isin(fraud_labels).all():
    raise ValueError("Unexpected synthetic fraud label")
transactions["is_fraud"] = raw_fraud.map(fraud_labels).astype("bool")
status_order = ("approved", "declined", "pending", "refunded")
if not set(transactions["status"]) <= set(status_order):
    raise ValueError("Unexpected transaction status")
transactions["status"] = transactions["status"].astype(
    pd.CategoricalDtype(categories=status_order, ordered=True)
)

In [ ]:
inspection = {
    "shape": transactions.shape,
    "columns": tuple(transactions.columns),
    "dtypes": transactions.dtypes.astype(str).to_dict(),
}
inspection

In [ ]:
status_counts = {
    str(label): int(count)
    for label, count in transactions["status"].value_counts().sort_index().items()
}
safe_counts = {
    "rows": len(transactions),
    "columns": len(transactions.columns),
    "fraud_rows": int(transactions["is_fraud"].sum()),
    "status_counts": status_counts,
}
safe_counts

In [ ]:
# Exercise 1 — missing-value analysis.
# Report a missing count for each column and explain which fields are optional.
raise NotImplementedError("Complete this project stage")

In [ ]:
# Exercise 2 — duplicate analysis.
# Compare full row duplicates with repeated transaction id values and explain the distinction.
raise NotImplementedError("Complete this project stage")

In [ ]:
# Exercise 3 — amount distribution.
# Summarize the amount distribution with selected quantile values and an appropriate plot.
raise NotImplementedError("Complete this project stage")

In [ ]:
# Exercise 4 — merchant and category aggregation.
# Aggregate transaction counts and amount totals, then present a deterministic ranking.
raise NotImplementedError("Complete this project stage")

In [ ]:
# Exercise 5 — temporal patterns.
# Compare transaction activity by UTC day and hour while preserving timezone awareness.
raise NotImplementedError("Complete this project stage")

In [ ]:
# Exercise 6 — fraud-class metrics.
# Explain class imbalance and calculate precision and recall for a clearly stated prediction rule.
raise NotImplementedError("Complete this project stage")

## Self-check

1. Why is `float64` acceptable at this EDA boundary but not in the domain model?
2. How did you distinguish missing optional data from invalid required data?
3. Which duplicate definition answers each business question?
4. Why must temporal grouping retain UTC awareness?
5. What does precision reveal that accuracy hides for rare fraud?
6. Are every output and execution count cleared before you save?